# Iterative optimization of a LAPACK function

## Loading from fortran

In [1]:
from peppo.ext.compilers import Flang
from peppo.ext.mlir import (
    MlirTranslate,
    FirOpt
)

FLANG_TOOL = Flang('/usr/bin/flang-new-22')
MLIR_TRANSLATE_TOOL = MlirTranslate()
FIR_OPT_TOOL = FirOpt()

print("Flang tool path:", FLANG_TOOL.path)
print("MLIR translate tool path:", MLIR_TRANSLATE_TOOL.path)
print("FIR optimize tool path:", FIR_OPT_TOOL.path)

Flang tool path: /usr/bin/flang-new-22
MLIR translate tool path: mlir-translate-22
FIR optimize tool path: fir-opt-22


In [2]:
import os

LAPACK_SOURCE_FILENAME = 'dgtsv.f95'
LAPACK_IR_FILENAME = 'dgtsv.ll'

with open(LAPACK_SOURCE_FILENAME, 'r') as file:
    src = file.read()

if not os.path.exists(LAPACK_IR_FILENAME) or False:
    lapack_llvm_ir = FLANG_TOOL.compile_to_llvm(src)

    with open(LAPACK_IR_FILENAME, 'w') as file:
        file.write(lapack_llvm_ir)
else:
    with open(LAPACK_IR_FILENAME, 'r') as file:
        lapack_llvm_ir = file.read()
        
print(lapack_llvm_ir)

; ModuleID = 'FIRModule'
source_filename = "FIRModule"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

$_QQclX444754535620 = comdat any

@_QFdgtsvECzero = internal constant double 0.000000e+00
@_QQclX444754535620 = linkonce constant [6 x i8] c"DGTSV ", comdat

define void @dgtsv_(ptr noalias %0, ptr noalias %1, ptr noalias %2, ptr noalias %3, ptr noalias %4, ptr noalias %5, ptr noalias %6, ptr noalias %7) {
  %9 = alloca double, i64 1, align 8
  %10 = alloca i32, i64 1, align 4
  %11 = alloca i32, i64 1, align 4
  %12 = alloca double, i64 1, align 8
  %13 = alloca i32, i64 1, align 4
  %14 = alloca i32, i64 1, align 4
  %15 = alloca [6 x i8], i64 1, align 1
  %16 = alloca i32, i64 1, align 4
  %17 = load i32, ptr %6, align 4
  %18 = sext i32 %17 to i64
  %19 = icmp sgt i64 %18, 0
  %20 = select i1 %19, i64 %18, i64 0
  store i32 0, ptr %7, align 4
  %21 = load i32, ptr %0, align 4
  %22 = icmp 

In [3]:
lapack_fir = FLANG_TOOL.compile_to_fir(src)
print(lapack_fir)

module attributes {dlti.dl_spec = #dlti.dl_spec<!llvm.ptr<270> = dense<32> : vector<4xi64>, !llvm.ptr<271> = dense<32> : vector<4xi64>, !llvm.ptr<272> = dense<64> : vector<4xi64>, i64 = dense<64> : vector<2xi64>, i128 = dense<128> : vector<2xi64>, f80 = dense<128> : vector<2xi64>, !llvm.ptr = dense<64> : vector<4xi64>, i1 = dense<8> : vector<2xi64>, i8 = dense<8> : vector<2xi64>, i16 = dense<16> : vector<2xi64>, i32 = dense<32> : vector<2xi64>, f16 = dense<16> : vector<2xi64>, f64 = dense<64> : vector<2xi64>, f128 = dense<128> : vector<2xi64>, "dlti.endianness" = "little", "dlti.mangling_mode" = "e", "dlti.legal_int_widths" = array<i32: 8, 16, 32, 64>, "dlti.stack_alignment" = 128 : i64>, fir.defaultkind = "a1c4d8i4l4r4", fir.kindmap = "", llvm.data_layout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128", llvm.ident = "Ubuntu flang version 22.1.8", llvm.target_triple = "x86_64-pc-linux-gnu"} {
  func.func @_QPdgtsv(%arg0: !fir.ref<i32> {fir.bindc_name

### Lifting LLVM IR into MLIR

In [4]:
lapack_lifted_mlir = MLIR_TRANSLATE_TOOL.translate_llvm_to_mlir(lapack_llvm_ir)

In [5]:
print(lapack_lifted_mlir)

"builtin.module"() ({
  "llvm.comdat"() <{sym_name = "__llvm_global_comdat"}> ({
    "llvm.comdat_selector"() <{comdat = 0 : i64, sym_name = "_QQclX444754535620"}> : () -> ()
  }) : () -> ()
  "llvm.mlir.global"() <{addr_space = 0 : i32, constant, dso_local, global_type = f64, linkage = #llvm.linkage<internal>, sym_name = "_QFdgtsvECzero", value = 0.000000e+00 : f64, visibility_ = 0 : i64}> ({
  }) : () -> ()
  "llvm.mlir.global"() <{addr_space = 0 : i32, comdat = @__llvm_global_comdat::@_QQclX444754535620, constant, global_type = !llvm.array<6 x i8>, linkage = #llvm.linkage<linkonce>, sym_name = "_QQclX444754535620", value = "DGTSV ", visibility_ = 0 : i64}> ({
  }) : () -> ()
  "llvm.module_flags"() <{flags = [#llvm.mlir.module_flag<warning, "Debug Info Version", 3 : i32>]}> : () -> ()
  "llvm.func"() <{CConv = #llvm.cconv<ccc>, arg_attrs = [{llvm.noalias}, {llvm.noalias}, {llvm.noalias}, {llvm.noalias}, {llvm.noalias}, {llvm.noalias}, {llvm.noalias}, {llvm.noalias}], function_type =

## Analysis with xDSL

In [6]:
import xdsl
from xdsl.context import Context
from xdsl.parser import Parser

In [7]:
xdsl.dialects.get_all_dialects()

{'acc': <function xdsl.dialects.get_all_dialects.<locals>.get_acc()>,
 'accfg': <function xdsl.dialects.get_all_dialects.<locals>.get_accfg()>,
 'affine': <function xdsl.dialects.get_all_dialects.<locals>.get_affine()>,
 'air': <function xdsl.dialects.get_all_dialects.<locals>.get_air()>,
 'arith': <function xdsl.dialects.get_all_dialects.<locals>.get_arith()>,
 'arm': <function xdsl.dialects.get_all_dialects.<locals>.get_arm()>,
 'arm_func': <function xdsl.dialects.get_all_dialects.<locals>.get_arm_func()>,
 'arm_neon': <function xdsl.dialects.get_all_dialects.<locals>.get_arm_neon()>,
 'asm': <function xdsl.dialects.get_all_dialects.<locals>.get_asm()>,
 'bigint': <function xdsl.dialects.get_all_dialects.<locals>.get_bigint()>,
 'bufferization': <function xdsl.dialects.get_all_dialects.<locals>.get_bufferization()>,
 'builtin': <function xdsl.dialects.get_all_dialects.<locals>.get_builtin()>,
 'cf': <function xdsl.dialects.get_all_dialects.<locals>.get_cf()>,
 'cmath': <function xdsl

In [8]:
from xdsl.dialects import (
    builtin,
    llvm,
    dlti,
    func,
    # fir
)

In [9]:
ctx = Context()
ctx.load_dialect(builtin.Builtin)
ctx.load_dialect(llvm.LLVM)
ctx.load_dialect(dlti.DLTI)
ctx.load_dialect(func.Func)

ctx.load_dialect(xdsl.dialects.get_all_dialects()['fir']())

ctx.allow_unregistered = True

### With vanilla MLIR

In [10]:
lapack_module = Parser(ctx, lapack_lifted_mlir).parse_module()

The first analysis I want to implement is about pointer types.
The arguments of the LAPACK function are pointers, I want to know how they are used.
The analysis I propose first enumerates how those pointers are used and then will make sure that the uses follow a consistent type.

In [11]:
for op in lapack_module.walk():
    print(op)

builtin.module attributes {dlti.dl_spec = #dlti.dl_spec<!llvm.ptr<270> = dense<32> : vector<4xi64>, !llvm.ptr<271> = dense<32> : vector<4xi64>, !llvm.ptr<272> = dense<64> : vector<4xi64>, i64 = dense<64> : vector<2xi64>, i128 = dense<128> : vector<2xi64>, f80 = dense<128> : vector<2xi64>, !llvm.ptr = dense<64> : vector<4xi64>, i1 = dense<8> : vector<2xi64>, i8 = dense<8> : vector<2xi64>, i16 = dense<16> : vector<2xi64>, i32 = dense<32> : vector<2xi64>, f16 = dense<16> : vector<2xi64>, f64 = dense<64> : vector<2xi64>, f128 = dense<128> : vector<2xi64>, "dlti.endianness" = "little", "dlti.mangling_mode" = "e", "dlti.legal_int_widths" = array<i32: 8, 16, 32, 64>, "dlti.stack_alignment" = 128 : i64>, llvm.ident = "Ubuntu flang version 22.1.8", llvm.module_asm = [], llvm.target_triple = "x86_64-pc-linux-gnu"} {
  "llvm.comdat"() <{sym_name = "__llvm_global_comdat"}> ({
    "llvm.comdat_selector"() <{comdat = 0 : i64, sym_name = "_QQclX444754535620"}> : () -> ()
  }) : () -> ()
  llvm.mlir.g

### With FIR

In [12]:
lapack_fir_generic = FIR_OPT_TOOL.optimize_fir(lapack_fir)
lapack_module = Parser(ctx, lapack_fir_generic).parse_module()